In [ ]:
#| default_exp train_flow

# Flow Matching Generative Model

Trains a flow matching model on pre-encoded (optionally PCA-reduced) embeddings.
Source distribution is N(0,I); target is the embedding distribution.
Uses RK4 integration and optional time warping at inference.

In [ ]:
#| export
import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import wandb
from midi_rae.data import EmbeddingDataset
from midi_rae.utils import *


In [ ]:
#| export
class VelocityNet(nn.Module):
    """MLP velocity field for flow matching.  Input: [x, (x_self_cond,) t_emb], output: dx/dt.
    Hidden layers use residual (skip) connections.
    t_dim: sinusoidal time embedding dim (replaces bare scalar t).
    self_condition: if True, also accepts x_self_cond (predicted x1 from prior pass); zeros when absent."""
    def __init__(self, input_dim, h_dim=256, n_layers=3, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.t_dim = t_dim
        net_in = input_dim * 2 + t_dim if self_condition else input_dim + t_dim
        self.fc_in  = nn.Linear(net_in, h_dim)
        self.hidden = nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
        self.fc_out = nn.Linear(h_dim, input_dim)

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = sinusoidal_time_emb(t, self.t_dim)          # [B, t_dim]
        if self.self_condition:
            sc = x_self_cond if x_self_cond is not None else torch.zeros_like(x)
            inp = torch.cat([x, sc, t_emb], dim=1)
        else:
            inp = torch.cat([x, t_emb], dim=1)
        h = F.gelu(self.fc_in(inp))
        for layer in self.hidden:
            h = F.gelu(layer(h)) + h
        return self.fc_out(h)


In [ ]:
#| export
import math

def sinusoidal_time_emb(t, dim=64):
    """Sinusoidal time embedding (à la DDPM/DiT).  t: [B] or [B,1] → [B, dim].
    Gives the model a rich multi-frequency view of t instead of a bare scalar."""
    if t.dim() > 1: t = t.squeeze(-1)
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32, device=t.device) / (half - 1))
    x = t.float().unsqueeze(1) * freqs.unsqueeze(0)   # [B, half]
    return torch.cat([x.sin(), x.cos()], dim=-1)       # [B, dim]


In [ ]:
#| export
class PerLevelFlowModel(nn.Module):
    """One VelocityNet per embedding level; each level's slice is routed to its own net.
    Has the same forward(x, t, x_self_cond=None) interface as VelocityNet.
    level_dims: list of ints, e.g. [20, 80, 320, 1280] from dataset.level_dims
    self_condition / t_dim: passed through to each VelocityNet.
    """
    def __init__(self, level_dims, h_dim=256, n_layers=4, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.nets = nn.ModuleList([VelocityNet(d, h_dim, n_layers, self_condition=self_condition, t_dim=t_dim)
                                   for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        outs, offset = [], 0
        for net, d in zip(self.nets, self.level_dims):
            sc_slice = x_self_cond[:, offset:offset+d] if x_self_cond is not None else None
            outs.append(net(x[:, offset:offset+d], t, sc_slice))
            offset += d
        return torch.cat(outs, dim=1)


In [ ]:
#| export
class CrossLevelFlowModel(nn.Module):
    """Per-patch flow model for coarse levels (L0, L1, L2).

    Each patch gets its own token (shared Linear weights within a level), so L2's 16 patches
    each have an independent token rather than being squashed into one.  The transformer
    cross-attends over all 1+4+16=21 patch tokens jointly.  No spatial conditioning — the
    fine model handles that for finer levels.

    level_dims:   flattened dims per level, e.g. [17, 124, 592]
    level_n_comp: PCA components per patch per level, e.g. [17, 31, 37]
                  n_patches per level is derived as level_dims[i] // level_n_comp[i]
    """
    def __init__(self, level_dims, level_n_comp, h_dim=512, n_layers=4, n_attn_layers=2,
                 n_heads=8, t_dim=64):
        super().__init__()
        self.level_dims   = list(level_dims)
        self.level_n_comp = list(level_n_comp)
        self.level_n_patches = [max(1, d // nc) for d, nc in zip(level_dims, level_n_comp)]
        self.t_dim = t_dim
        # Per-level input projection: shared across patches within a level
        self.level_in  = nn.ModuleList([nn.Linear(nc, h_dim) for nc in level_n_comp])
        # Time embedding added to every patch token
        self.t_proj = nn.Sequential(nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))
        # Cross-attention over all patch tokens (seq_len = sum of n_patches ≈ 21)
        enc_layer = nn.TransformerEncoderLayer(h_dim, n_heads, dim_feedforward=h_dim * 4,
                                               batch_first=True, dropout=0.0, norm_first=True)
        self.cross_attn = nn.TransformerEncoder(enc_layer, num_layers=n_attn_layers)
        # Per-level residual MLPs and output heads (shared across patches within level)
        self.level_mlp = nn.ModuleList([
            nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
            for _ in level_dims])
        self.level_out = nn.ModuleList([nn.Linear(h_dim, nc) for nc in level_n_comp])

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        B = x.size(0)
        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))  # [B, h_dim]
        # Build per-patch tokens for each level
        all_tokens, offset = [], 0
        for proj, nc, np_ in zip(self.level_in, self.level_n_comp, self.level_n_patches):
            d = np_ * nc
            xd = x[:, offset:offset+d].reshape(B, np_, nc)      # [B, P, nc]
            all_tokens.append(F.gelu(proj(xd)) + t_emb.unsqueeze(1))  # [B, P, h_dim]
            offset += d
        tokens = torch.cat(all_tokens, dim=1)                    # [B, total_patches, h_dim]
        tokens = self.cross_attn(tokens)
        # Per-level output: slice tokens, apply shared MLP+head, flatten
        outs, patch_offset = [], 0
        for mlp_layers, out_proj, np_, nc in zip(
                self.level_mlp, self.level_out, self.level_n_patches, self.level_n_comp):
            tok = tokens[:, patch_offset:patch_offset+np_, :]    # [B, P, h_dim]
            h = tok
            for layer in mlp_layers: h = F.gelu(layer(h)) + h
            outs.append(out_proj(h).reshape(B, np_ * nc))        # [B, P*nc]
            patch_offset += np_
        return torch.cat(outs, dim=1)

In [ ]:
#| export
class _ResBlock2d(nn.Module):
    """2D ResBlock: GroupNorm + SiLU + Conv, with FiLM conditioning. ch_in == ch_out."""
    def __init__(self, ch, cond_dim):
        super().__init__()
        ng = min(8, ch)
        self.net  = nn.Sequential(nn.GroupNorm(ng, ch), nn.SiLU(),
                                  nn.Conv2d(ch, ch, 3, padding=1),
                                  nn.GroupNorm(ng, ch), nn.SiLU(),
                                  nn.Conv2d(ch, ch, 3, padding=1))
        self.film = nn.Linear(cond_dim, ch * 2)

    def forward(self, x, cond):
        h = self.net[:3](x)
        γ, β = self.film(cond).chunk(2, dim=1)
        h = h * (1 + γ[:, :, None, None]) + β[:, :, None, None]
        h = self.net[3:](h)
        return x + h


class UNetFineFlowModel(nn.Module):
    """UNet velocity field for fine levels (L3@8x8, L4@16x16, L5@32x32).

    The UNet's spatial scales mirror the embedding hierarchy:
      enc scale 0: 32x32 <-> L5   enc scale 1: 16x16 <-> L4   enc scale 2: 8x8 <-> L3

    L4/L3 embeddings are injected at the matching encoder scale via 1x1 conv.
    Time + coarse conditioning are projected to a global FiLM vector at every block.
    Velocity heads branch off at each decoder scale for L3, L4, L5.

    Interface matches ConditionalFineFlowModel:
        forward(x_fine_flat, t, x_cond_flat) -> v_fine_flat
    """
    def __init__(self, cond_dims, target_dims, target_n_comp,
                 h_dim=64, t_dim=64, cond_n_comp=None):
        super().__init__()
        self.target_dims   = list(target_dims)
        self.target_n_comp = list(target_n_comp)
        self.t_dim = t_dim

        n_patches   = [d // nc for d, nc in zip(target_dims, target_n_comp)]
        self._grids = [int(n ** 0.5) for n in n_patches]   # [8, 16, 32]
        nc = target_n_comp   # [nc3, nc4, nc5]

        cd   = h_dim * 4   # FiLM conditioning dim
        self.cond_proj = nn.Sequential(
            nn.Linear(t_dim + sum(cond_dims), cd), nn.SiLU(), nn.Linear(cd, cd))

        chs         = [h_dim * m for m in [1, 2, 4, 4]]   # [h, 2h, 4h, 4h(bottle)]
        lateral_nc  = [nc[2], nc[1], nc[0]]               # L5 input, L4/L3 injections

        self.enc_in      = nn.Conv2d(nc[2], chs[0], 3, padding=1)
        self.enc_blocks  = nn.ModuleList()
        self.enc_injects = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        for i in range(3):
            self.enc_blocks.append(_ResBlock2d(chs[i], cd))
            self.enc_injects.append(
                nn.Conv2d(chs[i] + lateral_nc[i], chs[i], 1) if i > 0 else None)
            out_ch = chs[i+1] if i < 2 else chs[3]
            self.downsamples.append(nn.Conv2d(chs[i], out_ch, 3, stride=2, padding=1))
        self.bottleneck = _ResBlock2d(chs[3], cd)

        self.upsamples  = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        self.vel_heads  = nn.ModuleList()
        self.ch_reduces = nn.ModuleList()
        for i, (enc_i, up_ch) in enumerate([(2, chs[2]), (1, chs[1]), (0, chs[0])]):
            cat_ch = up_ch * 2
            self.upsamples.append(nn.Conv2d(up_ch, up_ch, 3, padding=1))
            self.dec_blocks.append(_ResBlock2d(cat_ch, cd))
            self.vel_heads.append(nn.Conv2d(cat_ch, nc[i], 1))
            if i < 2:
                self.ch_reduces.append(nn.Conv2d(cat_ch, chs[enc_i - 1], 1))

    def forward(self, x_flat, t, x_cond_flat):
        B = x_flat.shape[0]
        if t.dim() == 0:  t = t.unsqueeze(0).expand(B)
        elif t.dim() > 1: t = t.squeeze(-1)

        d3, d4, d5    = self.target_dims
        nc3, nc4, nc5 = self.target_n_comp
        g3, g4, g5    = self._grids
        laterals = [
            x_flat[:, d3+d4:  ].reshape(B, nc5, g5, g5),   # L5 — enc input
            x_flat[:, d3:d3+d4].reshape(B, nc4, g4, g4),   # L4 — injected at enc scale 1
            x_flat[:, :d3     ].reshape(B, nc3, g3, g3),   # L3 — injected at enc scale 2
        ]
        cond = self.cond_proj(torch.cat([sinusoidal_time_emb(t, self.t_dim), x_cond_flat], dim=1))

        h = F.silu(self.enc_in(laterals[0]))
        skips = []
        for i, (blk, inj, down) in enumerate(zip(self.enc_blocks, self.enc_injects, self.downsamples)):
            if inj is not None:
                h = inj(torch.cat([h, laterals[i]], dim=1))
            h = blk(h, cond)
            skips.append(h)
            h = down(h)
        h = self.bottleneck(h, cond)

        vels = []
        for i, (up, blk, head) in enumerate(zip(self.upsamples, self.dec_blocks, self.vel_heads)):
            h = up(F.interpolate(h, scale_factor=2, mode='nearest'))
            h = blk(torch.cat([h, skips[-(i+1)]], dim=1), cond)
            vels.append(head(h).flatten(1))
            if i < len(self.ch_reduces):
                h = self.ch_reduces[i](h)

        return torch.cat(vels, dim=1)   # [v_L3 | v_L4 | v_L5]


#| export
class FiLM(nn.Module):
    """Feature-wise Linear Modulation with pre-norm (AdaLN style).

    Applies adaptive LayerNorm: output = (1 + γ(cond)) * LN(x) + β(cond).
    Weights initialised to zero so the module starts as a plain LayerNorm
    (identity modulation), giving stable early training.
    """
    def __init__(self, cond_dim, feat_dim):
        super().__init__()
        self.norm  = nn.LayerNorm(feat_dim)
        self.gamma = nn.Linear(cond_dim, feat_dim)
        self.beta  = nn.Linear(cond_dim, feat_dim)
        nn.init.zeros_(self.gamma.weight); nn.init.zeros_(self.gamma.bias)
        nn.init.zeros_(self.beta.weight);  nn.init.zeros_(self.beta.bias)

    def forward(self, x, cond):
        return (1 + self.gamma(cond)) * self.norm(x) + self.beta(cond)


class ConditionalFineFlowModel(nn.Module):
    """Per-patch flow model for fine levels (L3, L4, L5).

    This model treats each patch as its own token and processes them with
    *shared* MLP weights — like a per-patch MLP applied in parallel.
    Conditioning comes from spatially-aligned parent tokens at each coarse level,
    looked up via precomputed parent indices (registered as buffers).

    Local inter-level attention (n_fine_attn_layers > 0) groups each fine parent
    with its children and applies a small transformer, giving bidirectional
    L3↔L4 and L4↔L5 attention within the fine hierarchy.

    Windowed lateral attention (n_lateral_attn_layers > 0) applies two passes of
    self-attention within spatial windows at each fine level (Swin-style): one on
    normal non-overlapping tiles, one on tiles shifted by (wH//2, wW//2) via cyclic
    roll. The two-pass scheme eliminates hard window-boundary seams, enabling
    coherent long notes spanning patch boundaries. Partition/unpartition uses pure
    reshape+permute (no fancy indexing) for torch.compile compatibility.

    Args:
        cond_dims:              flattened PCA dims for coarse levels
        target_dims:            flattened PCA dims for fine levels
        target_n_comp:          PCA components per fine patch
        h_dim:                  shared hidden dim
        n_layers:               number of pre-norm residual MLP blocks
        t_dim:                  sinusoidal time embedding dim
        cond_n_comp:            PCA components per coarse patch — int or list
        n_fine_attn_layers:     TransformerEncoder layers for inter-level attention
        n_lateral_attn_layers:  TransformerEncoder layers for windowed lateral attention
        lateral_window_size:    spatial window size (patches per side, default 4)
    """
    def __init__(self, cond_dims, target_dims, target_n_comp,
                 h_dim=256, n_layers=4, t_dim=64, cond_n_comp=[18, 24, 32],
                 n_fine_attn_layers=2, n_lateral_attn_layers=0, lateral_window_size=4,
                 grad_checkpoint=False):
        super().__init__()
        self.cond_dims     = list(cond_dims)
        self.target_dims   = list(target_dims)
        self.target_n_comp = list(target_n_comp)
        if isinstance(cond_n_comp, (int, float)):
            cond_n_comp = [int(cond_n_comp)] * len(cond_dims)
        self.cond_n_comp    = list(cond_n_comp)
        self.t_dim          = t_dim
        self.grad_checkpoint = grad_checkpoint

        cond_n_patches   = [max(1, d // nc) for d, nc in zip(cond_dims, self.cond_n_comp)]
        target_n_patches = [max(1, d // nc) for d, nc in zip(target_dims, target_n_comp)]
        cond_grids       = [self._square_grid(n) for n in cond_n_patches]
        target_grids     = [self._square_grid(n) for n in target_n_patches]
        self.target_n_patches = target_n_patches
        self._cond_n_patches  = cond_n_patches

        # Precompute coarse parent indices: pidx_{fi}_{ki} shape [n_fine_patches]
        for fi, (fH, fW) in enumerate(target_grids):
            for ki, (cH, cW) in enumerate(cond_grids):
                self.register_buffer(f'pidx_{fi}_{ki}', self._parent_idx(fH, fW, cH, cW))

        # Time embedding
        self.t_proj = nn.Sequential(
            nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))

        # Per-fine-level: project own patch [nc] → h_dim (shared across patches)
        self.patch_in = nn.ModuleList([nn.Linear(nc, h_dim) for nc in target_n_comp])

        # Per-fine-level × per-coarse-level: project parent [cnc] → h_dim
        self.parent_proj = nn.ModuleList([
            nn.ModuleList([nn.Linear(cnc, h_dim) for cnc in self.cond_n_comp])
            for _ in target_dims])

        # Learnable per-patch positional embeddings (zero-init = no bias at start)
        self.pos_emb = nn.ParameterList([
            nn.Parameter(torch.zeros(np, h_dim)) for np in target_n_patches])

        # Pre-norm residual MLP blocks, shared across all patches within a level
        self.mlp_blocks = nn.ModuleList([
            nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(h_dim),
                    nn.Linear(h_dim, h_dim * 4),
                    nn.GELU(),
                    nn.Linear(h_dim * 4, h_dim))
                for _ in range(n_layers)])
            for _ in target_dims])

        # Output head: h_dim → velocity [nc] per patch
        self.patch_out = nn.ModuleList([nn.Linear(h_dim, nc) for nc in target_n_comp])

        # Local inter-level attention: transformer over (parent, children) groups
        self.n_fine_attn_layers = n_fine_attn_layers
        if n_fine_attn_layers > 0 and len(target_grids) > 1:
            for fi in range(len(target_grids) - 1):
                pH, pW = target_grids[fi]
                cH, cW = target_grids[fi + 1]
                self.register_buffer(f'cidx_{fi}', self._child_idx(pH, pW, cH, cW))
            n_attn_heads = max(1, h_dim // 16)
            self.fine_pair_attn = nn.ModuleList([
                nn.TransformerEncoder(
                    nn.TransformerEncoderLayer(h_dim, n_attn_heads, dim_feedforward=h_dim*4,
                                               batch_first=True, dropout=0.0, norm_first=True),
                    num_layers=n_fine_attn_layers)
                for _ in range(len(target_grids) - 1)])
        else:
            self.fine_pair_attn = None

        # Windowed lateral self-attention: two passes (normal + shifted) Swin-style
        self.n_lateral_attn_layers = n_lateral_attn_layers
        if n_lateral_attn_layers > 0:
            n_attn_heads = max(1, h_dim // 16)
            self.lateral_attn = nn.ModuleList([
                nn.TransformerEncoder(
                    nn.TransformerEncoderLayer(h_dim, n_attn_heads, dim_feedforward=h_dim*4,
                                               batch_first=True, dropout=0.0, norm_first=True),
                    num_layers=n_lateral_attn_layers)
                for _ in target_dims])
            self.lateral_attn_shift = nn.ModuleList([
                nn.TransformerEncoder(
                    nn.TransformerEncoderLayer(h_dim, n_attn_heads, dim_feedforward=h_dim*4,
                                               batch_first=True, dropout=0.0, norm_first=True),
                    num_layers=n_lateral_attn_layers)
                for _ in target_dims])
            # Precompute window layout per fine level: (gH, gW, nH, nW, wH, wW)
            self._lateral_grids = []
            for gH, gW in target_grids:
                wH = min(lateral_window_size, gH)
                wW = min(lateral_window_size, gW)
                assert gH % wH == 0 and gW % wW == 0, \
                    f"Grid {gH}×{gW} not divisible by window {wH}×{wW}"
                self._lateral_grids.append((gH, gW, gH // wH, gW // wW, wH, wW))
        else:
            self.lateral_attn       = None
            self.lateral_attn_shift = None
            self._lateral_grids     = None

    @staticmethod
    def _square_grid(n):
        s = int(round(n ** 0.5))
        assert s * s == n, f"n_patches={n} is not a perfect square"
        return s, s

    @staticmethod
    def _parent_idx(fH, fW, cH, cW):
        """Index into coarse grid (cH×cW) for every patch in fine grid (fH×fW)."""
        r = torch.arange(fH).repeat_interleave(fW)
        c = torch.arange(fW).repeat(fH)
        return (r * cH // fH) * cW + (c * cW // fW)

    @staticmethod
    def _child_idx(pH, pW, cH, cW):
        """For each patch in parent grid (pH×pW), flat indices of its children in child grid (cH×cW)."""
        bH, bW = cH // pH, cW // pW
        r = torch.arange(pH).repeat_interleave(pW)
        c = torch.arange(pW).repeat(pH)
        children = [((r * bH + dr) * cW + (c * bW + dc)) for dr in range(bH) for dc in range(bW)]
        return torch.stack(children, dim=1)  # [pH*pW, bH*bW]

    def _attn(self, module, x):
        """Run module(x), using gradient checkpointing during training if enabled."""
        if self.grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(module, x, use_reentrant=False)
        return module(x)

    def forward(self, x_target, t, x_cond):
        """
        x_target: [B, sum(target_dims)]  noisy fine-level PCA embeddings
        t:        [B] or [B,1]           timestep
        x_cond:   [B, sum(cond_dims)]    coarse predicted endpoint x1 from first-stage flow
        Returns:  [B, sum(target_dims)]  predicted velocity
        """
        B = x_target.size(0)
        if t.dim() > 1: t = t.squeeze(-1)

        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))  # [B, h_dim]

        # Parse coarse levels → list of [B, n_coarse_patches, cnc]
        coarse, offset = [], 0
        for d, np, cnc in zip(self.cond_dims, self._cond_n_patches, self.cond_n_comp):
            coarse.append(x_cond[:, offset:offset+d].reshape(B, np, cnc))
            offset += d

        # Step 1: compute initial h for all fine levels (coarse conditioning + time + pos)
        hs, offset = [], 0
        for fi, (d, nc, np) in enumerate(
                zip(self.target_dims, self.target_n_comp, self.target_n_patches)):
            patches = x_target[:, offset:offset+d].reshape(B, np, nc)
            offset += d
            h = F.gelu(self.patch_in[fi](patches))
            h = h + self.pos_emb[fi]
            h = h + t_emb.unsqueeze(1)
            for ki, (coarse_emb, proj) in enumerate(zip(coarse, self.parent_proj[fi])):
                pidx = getattr(self, f'pidx_{fi}_{ki}')
                h = h + F.gelu(proj(coarse_emb[:, pidx, :]))
            hs.append(h)

        # Step 2: local inter-level attention (L3↔L4, L4↔L5, ...)
        if self.fine_pair_attn is not None:
            for fi, attn in enumerate(self.fine_pair_attn):
                cidx = getattr(self, f'cidx_{fi}')      # [n_parent, n_ch]
                n_parent, n_ch = cidx.shape
                h_p = hs[fi]        # [B, n_parent, h_dim]
                h_c = hs[fi + 1]    # [B, n_parent*n_ch, h_dim]
                group = torch.cat([h_p.unsqueeze(2), h_c[:, cidx, :]], dim=2).contiguous()
                group = self._attn(attn, group.reshape(B * n_parent, 1 + n_ch, -1))
                group = group.reshape(B, n_parent, 1 + n_ch, -1)
                hs[fi]     = group[:, :, 0, :].contiguous()
                h_c_new = hs[fi + 1].clone()
                h_c_new[:, cidx.reshape(-1), :] = group[:, :, 1:, :].reshape(B, n_parent * n_ch, -1)
                hs[fi + 1] = h_c_new

        # Step 2.5: windowed lateral self-attention (normal windows)
        if self.lateral_attn is not None:
            for fi, (lat_attn, (gH, gW, nH, nW, wH, wW)) in enumerate(
                    zip(self.lateral_attn, self._lateral_grids)):
                h = hs[fi]  # [B, gH*gW, h_dim]
                h = h.reshape(B, nH, wH, nW, wW, -1)
                h = h.permute(0, 1, 3, 2, 4, 5).contiguous()   # [B, nH, nW, wH, wW, h_dim]
                h = h.reshape(B * nH * nW, wH * wW, -1)         # [B*n_win, win_seq, h_dim]
                h = self._attn(lat_attn, h)
                h = h.reshape(B, nH, nW, wH, wW, -1)
                h = h.permute(0, 1, 3, 2, 4, 5).contiguous()   # [B, nH, wH, nW, wW, h_dim]
                hs[fi] = h.reshape(B, gH * gW, -1)

        # Step 2.6: shifted-window lateral attention (Swin-style, removes hard seams)
        if self.lateral_attn_shift is not None:
            for fi, (lat_attn, (gH, gW, nH, nW, wH, wW)) in enumerate(
                    zip(self.lateral_attn_shift, self._lateral_grids)):
                h = hs[fi].reshape(B, gH, gW, -1)
                h = torch.roll(h, shifts=(wH // 2, wW // 2), dims=(1, 2))
                h = h.reshape(B, nH, wH, nW, wW, -1)
                h = h.permute(0, 1, 3, 2, 4, 5).contiguous()
                h = h.reshape(B * nH * nW, wH * wW, -1)
                h = self._attn(lat_attn, h)
                h = h.reshape(B, nH, nW, wH, wW, -1)
                h = h.permute(0, 1, 3, 2, 4, 5).contiguous()
                h = h.reshape(B, gH, gW, -1)
                h = torch.roll(h, shifts=(-wH // 2, -wW // 2), dims=(1, 2))
                hs[fi] = h.reshape(B, gH * gW, -1)

        # Step 3: MLP blocks + output
        velocities = []
        for fi, h in enumerate(hs):
            for block in self.mlp_blocks[fi]:
                h = h + block(h)
            velocities.append(self.patch_out[fi](h).reshape(B, -1))
        return torch.cat(velocities, dim=1)


In [ ]:
#| export
def warp_time(t, s=0.5):
    """Parametric time warping (Scott H. Hawley, 'Flow With What You Know', ICLR 2025).
    s=1 → linear; s<1 → slower near middle; s=1.5 ≈ cosine schedule.
    Works on scalar, 1-D or 2-D tensors."""
    return 4*(1-s)*t**3 + 6*(s-1)*t**2 + (3-2*s)*t

def sample_time(shape, schedule='warp', warp_s=0.5, sine_kappa=0.0, device='cpu'):
    """Sample flow timesteps t∈[0,1] with schedule warping.
    schedule='kappa_sine' → t = (1-κ)·τ + κ·sin(π/2·τ); κ=sine_kappa; upweights t near 1
    schedule='sine'       → t = sin(π/2·τ); density ∝ 1/√(1-t²), concentrates near t=1
    schedule='warp'       → polynomial warp_time (default)
    schedule='linear'     → uniform, no warping
    """
    tau = torch.rand(*shape, device=device)
    if schedule == 'kappa_sine':
        return (1 - sine_kappa) * tau + sine_kappa * torch.sin(math.pi / 2 * tau)
    elif schedule == 'sine':
        return torch.sin(math.pi / 2 * tau)
    elif schedule == 'warp':
        return warp_time(tau, s=warp_s)
    else:
        return tau

In [ ]:
#| export
@torch.no_grad()
def rk4_step(model, y, t, dt):
    """4th-order Runge-Kutta step for the learned velocity field."""
    t_  = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    k1 = model(y,             t_)
    k2 = model(y + dt*k1/2,   t_ + dt/2)
    k3 = model(y + dt*k2/2,   t_ + dt/2)
    k4 = model(y + dt*k3,     t_ + dt)
    return y + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

@torch.no_grad()
def euler_step(model, y, t, dt):
    t_ = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    return y + model(y, t_) * dt

In [ ]:
#| export
def sample_source(shape, device='cpu', source_df=None, source_scales=None, level_dims=None):
    """Sample from source distribution with optional per-level Student-t and scaling.
    source_df: scalar df → Student-t for all dims; list → per-level (None/0 = Gaussian, float = Student-t)
    source_scales: list of per-level scale factors applied after sampling
    """
    if isinstance(source_df, (list, tuple)):
        # Per-level: each slice sampled independently
        assert level_dims is not None, "level_dims required for per-level source_df"
        batch = shape[:-1]
        y = torch.empty(*shape, device=device)
        offset = 0
        for df, d in zip(source_df, level_dims):
            sl = (*batch, d)
            if df:
                normal = torch.randn(*sl, device=device)
                gamma  = torch._standard_gamma(torch.full(sl, df/2, device=device)) / (df/2)
                y[..., offset:offset+d] = normal / gamma.sqrt()
            else:
                y[..., offset:offset+d] = torch.randn(*sl, device=device)
            offset += d
    elif source_df:
        # Scalar df → Student-t for all dims
        normal = torch.randn(*shape, device=device)
        gamma  = torch._standard_gamma(torch.full(shape, source_df/2, device=device)) / (source_df/2)
        y = normal / gamma.sqrt()
    else:
        y = torch.randn(*shape, device=device)
    if source_scales is not None and level_dims is not None:
        offset = 0
        for scale, d in zip(source_scales, level_dims):
            y[..., offset:offset+d] *= scale
            offset += d
    return y

In [ ]:
#| export
@torch.no_grad()
def generate_samples_conditional(coarse_model, fine_model, n_samples, coarse_dim,
                                 target_dims, device='cpu', n_steps=20,
                                 step_fn=rk4_step, warp_s=0.5,
                                 coarse_source_df=None, coarse_source_scales=None,
                                 coarse_level_dims=None, fine_source_scales=None,
                                 use_diffeq_fine=False):
    """Two-stage conditional sampler: coarse flow → fine conditional flow.

    Conditioning signal: x1_pred_coarse = x_t_coarse + (1-t) * v_coarse
    — the coarse model's predicted endpoint at each step.  This is more
    informative than the noisy state x_t_coarse alone (especially at small t
    where x_t is mostly noise), and is in the same space as the final coarse
    embeddings, normalising out the t-dependence of the velocity scale.

    Both models share the *same* timestep grid — enforced by construction.
    The coarse model is called once per step to get v_coarse for x1_pred,
    then step_fn (which may call it again internally for RK4) advances the
    coarse state.  The fine model uses a simple Euler step conditioned on
    x1_pred_coarse computed at the start of each step.

    use_diffeq_fine=True: run coarse RK4 to completion first, then solve the
    fine ODE with torchdiffeq/dopri5 conditioned on the final coarse output.
    Matches training distribution better when fine_teacher_forcing=True (where
    x_cond = real_coarse, the actual final output, not an intermediate estimate).

    Training counterpart: at each batch, freeze coarse model, compute
    x1_pred_coarse = x_t_coarse + (1-t)*coarse_model(x_t_coarse, t),
    then train fine_model(x_t_fine, t, x1_pred_coarse) with flow-matching loss.

    Args:
        coarse_model:        first-stage flow (CrossLevelFlowModel / PerLevelFlowModel)
        fine_model:          ConditionalFineFlowModel
        coarse_dim:          total dim of coarse-level output (sum of PCA dims L0-L3)
        target_dims:         list of flattened dims for fine levels (e.g. [D_L4, D_L5])
        coarse_source_*:     source distribution kwargs forwarded to the coarse stage
        fine_source_scales:  optional per-level scale factors for fine-level noise
        use_diffeq_fine:     use torchdiffeq/dopri5 for fine ODE (coarse is always RK4)
    Returns:
        coarse_out : [n_samples, coarse_dim]
        fine_out   : [n_samples, sum(target_dims)]
    """
    fine_dim = sum(target_dims)
    y_coarse = sample_source((n_samples, coarse_dim), device=device,
                             source_df=coarse_source_df,
                             source_scales=coarse_source_scales,
                             level_dims=coarse_level_dims)
    y_fine = sample_source((n_samples, fine_dim), device=device,
                           source_scales=fine_source_scales,
                           level_dims=target_dims)
    ts = warp_time(torch.linspace(0, 1, n_steps + 1), s=warp_s)
    coarse_model.eval(); fine_model.eval()
    for i in range(n_steps):
        dt   = (ts[i+1] - ts[i]).item()
        t_s  = ts[i].item()
        t    = torch.full((n_samples, 1), t_s, device=device)

        # x1_pred_coarse: coarse model's predicted endpoint at current state/time
        v_coarse_pred  = coarse_model(y_coarse, t)
        x1_pred_coarse = y_coarse + (1 - t_s) * v_coarse_pred  # [n_samples, coarse_dim]

        # Advance coarse state (step_fn may call coarse_model again internally for RK4)
        y_coarse = step_fn(coarse_model, y_coarse, t_s, dt)

        if not use_diffeq_fine:
            # Fine model conditioned on x1_pred_coarse; Euler step
            v_fine = fine_model(y_fine, t, x1_pred_coarse)
            y_fine = y_fine + v_fine * dt

    if use_diffeq_fine:
        # Solve fine ODE with adaptive dopri5, conditioned on final coarse output
        from torchdiffeq import odeint
        coarse_out = y_coarse
        def fine_func(t_scalar, y):
            t_ = t_scalar.expand(y.size(0), 1)
            return fine_model(y, t_, coarse_out)
        t_span = torch.tensor([0.0, 1.0], device=device)
        y_fine = odeint(fine_func, y_fine, t_span, method='dopri5', rtol=1e-4, atol=1e-4)[1]

    return y_coarse, y_fine


In [ ]:
#| eval: false
# Smoke test: ConditionalFineFlowModel forward pass
# cond: 3 coarse levels with 20 comp/patch → [1,4,16] patches (1×1, 2×2, 4×4 grids)
# target: 2 fine levels with [4,3] comp/patch → [16,64] patches (4×4, 8×8 grids)
import torch
cond_dims    = [20, 80, 320]          # 1,4,16 coarse patches × 20 comp
target_dims  = [64, 192]              # 16,64 fine patches × 4,3 comp
target_n_comp = [4, 3]
m = ConditionalFineFlowModel(cond_dims, target_dims, target_n_comp, h_dim=64, n_layers=3)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
B = 4
v = m(torch.randn(B, sum(target_dims)), torch.rand(B), torch.randn(B, sum(cond_dims)))
assert v.shape == (B, sum(target_dims)), f"Expected {(B, sum(target_dims))}, got {v.shape}"
print(f'Output shape: {v.shape}  OK')
# Verify parent indices (L1 has 4 patches in 2×2 grid; L2 fine patch 0 at (0,0) → parent (0,0)=0)
assert m.pidx_1_1[0].item() == 0
print('Parent index check OK')

In [ ]:
#| export
def ann_repair(source, target, n_projections=1, chunk_size=None):
    """Approximate nearest-neighbor re-pairing of source and target batches.

    Sorts both source and target by their projection onto random unit vectors
    and pairs by rank — equivalent to exact 1-D OT along that direction.
    Fully on-device (GPU-friendly), O(B log B) per projection.

    chunk_size: if set, processes the batch in chunks of this size and repairs
    independently within each chunk. Smaller chunks are faster but less optimal;
    default (None) processes the whole batch at once.

    With n_projections > 1, tries multiple random directions and keeps the
    pairing with the lowest total squared transport cost.

    Args:
        source:        (B, D) tensor on any device
        target:        (B, D) tensor on same device
        n_projections: number of random projections to try per chunk
        chunk_size:    chunk size for within-batch processing (None = full batch)

    Returns:
        (source_repaired, target_repaired): re-ordered so source[i] ↔ target[i]
        approximately minimises total squared transport cost.
    """
    B, D = source.shape
    C = B if (chunk_size is None or chunk_size >= B) else chunk_size

    s_out = torch.empty_like(source)
    t_out = torch.empty_like(target)
    for start in range(0, B, C):
        end  = min(start + C, B)
        s, t = source[start:end], target[start:end]
        best_s, best_t, best_cost = s, t, float('inf')
        for _ in range(n_projections):
            proj   = torch.randn(D, device=s.device, dtype=s.dtype)
            proj   = proj / proj.norm()
            s_rep  = s[(s @ proj).argsort()]
            t_rep  = t[(t @ proj).argsort()]
            cost   = (s_rep - t_rep).pow(2).sum().item()
            if cost < best_cost:
                best_cost = cost
                best_s, best_t = s_rep, t_rep
        s_out[start:end] = best_s
        t_out[start:end] = best_t
    return s_out, t_out


def do_pairing(source, target, method='ann', n_projections=1, chunk_size=None):
    """Unified source-target pairing wrapper.

    method='ann':   approximate 1-D OT via random projections (fast, GPU-friendly)
    method='exact': exact OT via POT's Earth Mover's Distance solver (O(n^3), CPU)
    method='none':  no repairing (random pairing)
    """
    if method == 'none':
        return source, target
    if method == 'exact':
        try:
            import ot as pot
            import numpy as np
            x0 = source.reshape(source.shape[0], -1).float()
            x1 = target.reshape(target.shape[0], -1).float()
            a  = pot.unif(x0.shape[0])
            b  = pot.unif(x1.shape[0])
            M  = torch.cdist(x0.cpu(), x1.cpu()).pow(2).numpy()
            pi = pot.emd(a, b, M)
            i_s, i_t = np.divmod(np.random.choice(pi.size, p=(pi/pi.sum()).flatten(),
                                                   size=source.shape[0], replace=False),
                                  pi.shape[1])
            return source[i_s], target[i_t]
        except ImportError:
            print("WARNING: POT not installed, falling back to ann pairing")
    return ann_repair(source, target, n_projections=n_projections, chunk_size=chunk_size)


In [ ]:
#| export
@torch.no_grad()
def generate_samples(model, n_samples, dim, device='cpu',
                     n_steps=20, step_fn=rk4_step, warp_s=0.5, source_df=None,
                     source_scales=None, level_dims=None):
    """Sample from the flow model: integrate noise → embedding space."""
    y = sample_source((n_samples, dim), device=device, source_df=source_df,
                      source_scales=source_scales, level_dims=level_dims)
    ts = torch.linspace(0, 1, n_steps + 1)
    ts = warp_time(ts, s=warp_s)
    model.eval()
    for i in range(n_steps):
        dt = (ts[i+1] - ts[i]).item()
        y  = step_fn(model, y, ts[i].item(), dt)
    return y

In [ ]:
#| export
@torch.no_grad()
def generate_samples_diffeq(model, n_samples, dim, device='cpu',
                             source_df=None, source_scales=None, level_dims=None,
                             rtol=1e-5, atol=1e-5, method='dopri5'):
    """Adaptive ODE sampler via torchdiffeq (default: dopri5).
    Automatically concentrates evaluations where the velocity field is stiff
    (around t≈0.85-0.93 for our coarse model) — no manual time warping needed.
    For unconditional models only (coarse); fine model uses generate_samples_conditional.
    """
    from torchdiffeq import odeint
    y0 = sample_source((n_samples, dim), device=device, source_df=source_df,
                       source_scales=source_scales, level_dims=level_dims)
    model.eval()
    def func(t, y):
        t_ = t.expand(y.size(0), 1)
        return model(y, t_)
    t_span = torch.tensor([0.0, 1.0], device=device)
    return odeint(func, y0, t_span, method=method, rtol=rtol, atol=atol)[1]

In [ ]:
#| export
def mmd_rbf(x, y, n_sub=2000):
    """Unbiased MMD² with RBF kernel, median bandwidth heuristic.
    x, y: (N, D) tensors. Subsamples to n_sub for speed."""
    if x.size(0) > n_sub: x = x[torch.randperm(x.size(0))[:n_sub]]
    if y.size(0) > n_sub: y = y[torch.randperm(y.size(0))[:n_sub]]
    xy = torch.cat([x, y], dim=0)
    sigma2 = torch.cdist(xy, xy).median().pow(2).clamp(min=1e-6)
    def rbf(a, b): return torch.exp(-torch.cdist(a, b).pow(2) / (2 * sigma2))
    return (rbf(x, x).mean() + rbf(y, y).mean() - 2 * rbf(x, y).mean()).item()


In [ ]:
#| export
def wasserstein_score(x, y, n_projections=200, n_sub=2000):
    """Sliced Wasserstein distance: average 1-D Wasserstein over random projections.
    Falls back gracefully if geomloss is unavailable.
    Returns nan on numerical failure (overflow, diverged samples, etc.).
    x, y: (N, D) numpy arrays. Blur is auto-scaled to median pairwise distance."""
    try:
        import geomloss
        xt = torch.tensor(x[:n_sub]).float()
        yt = torch.tensor(y[:n_sub]).float()
        scale = torch.cdist(xt[:256], xt[:256]).median().clamp(min=1e-3).item()
        loss = geomloss.SamplesLoss("sinkhorn", p=2, blur=0.05 * scale)
        return loss(xt, yt).item()
    except ImportError:
        pass
    except Exception:
        return float('nan')
    try:
        from scipy.stats import wasserstein_distance
        rng = np.random.default_rng(0)
        D = x.shape[1]
        projs = rng.standard_normal((D, n_projections))
        projs /= np.linalg.norm(projs, axis=0, keepdims=True)
        px, py = x[:n_sub] @ projs, y[:n_sub] @ projs
        return float(np.mean([wasserstein_distance(px[:, i], py[:, i]) for i in range(n_projections)]))
    except Exception:
        return float('nan')

In [ ]:
#| export
@torch.no_grad()
def eval_flow(model, real_embeddings, n_samples=10000, n_steps=20, warp_s=0.5, device='cpu',
              source_df=None, source_scales=None, level_dims=None, gen=None, level_names=None,
              use_diffeq=True, rtol=1e-5, atol=1e-5, level_n_patches=None):
    """Compare distributional statistics of real vs generated embeddings, per level.
    Returns flat dict with keys like 'L0/mmd', 'L0/wasserstein', 'L0/real_std', etc.
    Also returns global 'mmd' and 'wasserstein' for backward compatibility.
    real_embeddings: (N, D) tensor.
    gen: optional pre-computed generated samples (N, D) tensor — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' keys.
    level_n_patches: list of ints, one per level. If n_patches>1, metrics are computed per patch
                     (reshape (N, n_patches*n_comp) → (N*n_patches, n_comp)) for reliability.
    use_diffeq: use adaptive dopri5 solver (torchdiffeq) instead of fixed-step RK4.
    """
    from scipy.stats import skew, kurtosis
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    if gen is None:
        dim = real_embeddings.shape[1]
        try:
            if use_diffeq:
                gen = generate_samples_diffeq(model, n_samples, dim, device=device,
                                              source_df=source_df, source_scales=source_scales,
                                              level_dims=level_dims, rtol=rtol, atol=atol).cpu()
            else:
                raise ImportError  # fall through to RK4
        except (ImportError, Exception) as e:
            if use_diffeq: print(f"  torchdiffeq unavailable ({e}), falling back to RK4")
            gen = generate_samples(model, n_samples, dim, device=device,
                                   n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                                   source_scales=source_scales, level_dims=level_dims).cpu()
    else:
        gen = gen[:n_samples].float().cpu()
    r, g = real.numpy(), gen.numpy()

    metrics = {}
    metrics['real_mean']  = float(r.mean())
    metrics['real_std']   = float(r.std())
    metrics['real_skew']  = float(skew(r.ravel()))
    metrics['real_kurt']  = float(kurtosis(r.ravel()))
    metrics['gen_mean']   = float(g.mean())
    metrics['gen_std']    = float(g.std())
    metrics['gen_skew']   = float(skew(g.ravel()))
    metrics['gen_kurt']   = float(kurtosis(g.ravel()))
    metrics['mmd']        = mmd_rbf(real, gen)
    metrics['wasserstein'] = wasserstein_score(r, g)

    if level_dims is not None:
        offset = 0
        for i, d in enumerate(level_dims):
            rl = r[:, offset:offset+d]
            gl = g[:, offset:offset+d]
            n_patches = level_n_patches[i] if level_n_patches else 1
            if n_patches > 1:
                n_comp = d // n_patches
                rl = rl.reshape(-1, n_comp)
                gl = gl.reshape(-1, n_comp)
            rt, gt = torch.tensor(rl), torch.tensor(gl)
            lname = level_names[i] if level_names else f'L{i}'
            metrics[f'{lname}/real_std']    = float(rl.std())
            metrics[f'{lname}/gen_std']     = float(gl.std())
            metrics[f'{lname}/real_kurt']   = float(kurtosis(rl.ravel()))
            metrics[f'{lname}/gen_kurt']    = float(kurtosis(gl.ravel()))
            metrics[f'{lname}/mmd']         = mmd_rbf(rt, gt)
            metrics[f'{lname}/wasserstein'] = wasserstein_score(rl, gl)
            offset += d

    w = max(len(k) for k in metrics)
    for k, v in metrics.items():
        print(f'  {k:{w}s} = {v:.4f}')
    return metrics

In [ ]:
#| export
def eval_jacobian_norm_vs_t(model, x_sample, n_t=20, n_epsilon=4, cond=None, device='cpu'):
    """Estimate Frobenius norm of the Jacobian dv/dx as a function of t.

    Uses Hutchinson estimator: E_ε[||J^T ε||²] = ||J||²_F  with Rademacher ε.
    A peak in the curve at some t* reveals where the flow is making its hardest
    topological decisions (routing mass to disjoint clusters).

    Args:
        model:     velocity field; called as model(x, t) or model(x, t, cond)
        x_sample:  (B, D) batch of real data points
        n_t:       number of time steps to sweep
        n_epsilon: number of Rademacher samples per t (more = lower variance)
        cond:      optional conditioning tensor (B, cond_dim) for conditional models
        device:    compute device
    Returns:
        t_vals (list[float]), norm_vals (list[float])
    """
    model.eval()
    x_sample = x_sample.to(device).float()
    t_vals, norm_vals = [], []

    for t_val in torch.linspace(0.01, 0.99, n_t).tolist():
        t = torch.full((len(x_sample), 1), t_val, device=device)
        x = x_sample.detach().requires_grad_(True)

        est_list = []
        for _ in range(n_epsilon):
            epsilon = (torch.randint_like(x, low=0, high=2).float() * 2 - 1)  # Rademacher ±1
            if cond is not None:
                v = model(x, t, cond.to(device))
            else:
                v = model(x, t)
            vjp = torch.autograd.grad(v, x, grad_outputs=epsilon,
                                       retain_graph=True, create_graph=False)[0]
            # Frobenius norm estimator: E[||J^T ε||²] = ||J||²_F
            est_list.append(vjp.pow(2).sum(dim=-1))   # (B,)

        norm_sq = torch.stack(est_list).mean(0).mean().item()   # scalar
        t_vals.append(t_val)
        norm_vals.append(norm_sq ** 0.5)              # sqrt for Frobenius norm

    return t_vals, norm_vals

In [ ]:

#| export
@torch.no_grad()
def plot_level_histograms(model, real_embeddings, level_dims, n_samples=10000,
                          n_steps=20, warp_s=0.5, device='cpu', n_bins=100, source_df=None,
                          source_scales=None, epoch=None, gen=None, level_names=None):
    """Return dict of per-level histogram figures {'L0': fig, 'L1': fig, ...}.
    level_dims: list of ints, flattened PCA dims per level e.g. [20, 80, 320, 1280]
    real_embeddings: (N, sum(level_dims)) tensor
    gen: optional pre-computed generated samples — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' labels.
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float().numpy()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu().numpy()
    else:
        gen = gen[:n_samples].float().cpu().numpy()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d].ravel()
        g = gen[:,  offset:offset+d].ravel()
        lim = np.percentile(np.abs(np.concatenate([r, g])), 99)
        bins = np.linspace(-lim, lim, n_bins + 1)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(r, bins=bins, alpha=0.5, color='steelblue', label='real', density=True)
        ax.hist(g, bins=bins, alpha=0.5, color='darkorange', label='gen',  density=True)
        lname = level_names[i] if level_names else f'L{i}'
        title = f'{lname} ({d}d)'
        if epoch is not None: title += f' — Epoch {epoch}'
        ax.set_title(title)
        ax.set_xlabel('value')
        ax.legend(fontsize=8)
        plt.tight_layout()
        figs[lname] = fig
        offset += d
    return figs

In [ ]:
#| export
@torch.no_grad()
def plot_level_scatter(model, real_embeddings, level_dims, n_samples=5000,
                       n_steps=20, warp_s=0.5, device='cpu',
                       source_df=None, source_scales=None, epoch=None,
                       gen=None, level_names=None, level_n_components=None,
                       pca_cache=None):
    """Return dict of per-level 3D PCA scatter plots {'L0/real': fig, 'L0/gen': fig, ...}.
    gen: optional pre-computed generated samples — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' labels.
    level_n_components: optional list of ints (one per level). If provided, each level's data is
        reshaped from (B, n_patches × n_comp) → (B × n_patches, n_comp) before PCA so that each
        patch is a point — matching the encoder training viz style (make_emb_viz / _gather_level).
        After reshape, subsampled back to n_samples points to keep plots manageable.
    pca_cache: optional dict {lname: fitted SklearnPCA}. If provided, cached PCAs are reused
        instead of refitting, and newly fitted PCAs are stored into it. Pass the same dict
        across epochs to keep all gen scatters in the same coordinate frame as the first real scatter.
    """
    from midi_rae.viz import plot_embeddings_3d
    from sklearn.decomposition import PCA as SklearnPCA
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu()
    else:
        gen = gen[:n_samples].float().cpu()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d]
        g = gen[:,  offset:offset+d]
        lname = level_names[i] if level_names else f'L{i}'
        title_sfx = f' — Epoch {epoch}' if epoch is not None else ''
        if level_n_components is not None:
            n_comp = level_n_components[i]
            n_patches = d // n_comp
            r = r.reshape(-1, n_comp)   # (B × n_patches, n_comp) — each patch is a point
            g = g.reshape(-1, n_comp)
            # subsample after reshape so scatter stays at ~n_samples points
            if r.shape[0] > n_samples:
                sub = torch.randperm(r.shape[0])[:n_samples]
                r = r[sub]
                g = g[sub]
        r_np = r.float().numpy() if isinstance(r, torch.Tensor) else r
        g_np = g.float().numpy() if isinstance(g, torch.Tensor) else g
        if len(r_np) >= 4:
            # Reuse cached PCA if available; otherwise fit on real and cache it
            if pca_cache is not None and lname in pca_cache:
                pca3 = pca_cache[lname]
            else:
                pca3 = SklearnPCA(n_components=3).fit(r_np)
                if pca_cache is not None:
                    pca_cache[lname] = pca3
            r3 = pca3.transform(r_np)
            g3 = pca3.transform(g_np)
            figs[f'{lname}/real'] = plot_embeddings_3d(r3, color_by='random', title=f'{lname} ({n_comp if level_n_components else d}d/patch) real{title_sfx}')
            figs[f'{lname}/gen']  = plot_embeddings_3d(g3, color_by='random', title=f'{lname} ({n_comp if level_n_components else d}d/patch) gen{title_sfx}')
        offset += d
    return figs

In [ ]:
#| export
def _wandb_log_viz(log_dict, eval_model, embeddings, level_dims, epoch,
                   real_scatter_logged, gen=None, level_names=None,
                   device='cpu', warp_s=0.5, source_df=None, source_scales=None,
                   level_n_components=None, pca_cache=None):
    """Log per-level histograms and 3-D scatter plots to W&B via log_dict.
    gen: optional pre-computed generated samples — avoids redundant forward passes.
    level_n_components: passed to plot_level_scatter to show per-patch points (encoder viz style).
    pca_cache: dict {lname: SklearnPCA} shared across epochs — ensures all gen scatters use the
        same projection as the first real scatter. Pass {} on first call; it is mutated in-place.
    Modifies log_dict in-place. Returns updated real_scatter_logged flag."""
    import wandb, matplotlib.pyplot as plt
    figs = plot_level_histograms(eval_model, embeddings, level_dims, epoch=epoch,
                                 gen=gen, level_names=level_names, device=device,
                                 warp_s=warp_s, source_df=source_df, source_scales=source_scales)
    for lname, fig in figs.items():
        log_dict[f'media/hist_{lname}'] = wandb.Image(fig, caption=f'Epoch {epoch}')
        plt.close(fig)
    del figs
    scatters = plot_level_scatter(eval_model, embeddings, level_dims, epoch=epoch,
                                  gen=gen, level_names=level_names, device=device,
                                  warp_s=warp_s, source_df=source_df, source_scales=source_scales,
                                  level_n_components=level_n_components, pca_cache=pca_cache)
    for lname, fig in scatters.items():
        if lname.endswith('/real') and real_scatter_logged:
            fig.data = []
            continue
        log_dict[f'media/scatter_{lname.replace("/", "_")}'] = wandb.Html(fig.to_html())
    del scatters
    return True  # real_scatter_logged

In [ ]:
#| export
def _wandb_log_jacobian(log_dict, eval_model, sample_data, n_t=20, n_epsilon=4,
                        cond=None, device='cpu'):
    """Evaluate Jacobian Frobenius norm vs t and log a line chart to W&B via log_dict.
    Modifies log_dict in-place.
    cond: optional conditioning tensor (B, cond_dim) for conditional models."""
    import wandb
    t_vals, jac_norms = eval_jacobian_norm_vs_t(
        eval_model, sample_data, n_t=n_t, n_epsilon=n_epsilon, cond=cond, device=device)
    jac_table = wandb.Table(columns=['t', 'jacobian_norm'],
                            data=[[t, n] for t, n in zip(t_vals, jac_norms)])
    log_dict['eval/jacobian_norm_vs_t'] = wandb.plot.line(
        jac_table, 't', 'jacobian_norm', title='Jacobian Frobenius Norm vs t')

In [ ]:
#| export
@torch.no_grad()
def decode_flow_to_piano_rolls(coarse_emb, fine_emb, pca_models,
                                coarse_level_dims, fine_level_dims, fine_levels_idx,
                                cfg, decoder, device, n_samples=16,
                                coarse_n_patches=None, fine_n_patches=None):
    """Decode flow-generated coarse + fine embeddings to piano rolls (no HMEP).
    coarse_emb:      (B, sum_coarse_dims) — PCA-compressed or raw coarse embeddings
    fine_emb:        (B, sum_fine_dims)   — PCA-compressed or raw fine embeddings
    pca_models:      dict {level_idx: sklearn PCA} or None. If None or level missing,
                     embeddings are treated as already in full embedding space.
    coarse_n_patches: list of n_patches per coarse level (required for raw mode)
    fine_n_patches:   list of n_patches per fine level (required for raw mode)
    """
    from midi_rae.generate import build_patch_states, batch_patch_states, build_enc_out, make_grid_pos, binarize
    from midi_rae.core import PatchState

    B = min(n_samples, coarse_emb.shape[0])
    coarse_emb = coarse_emb[:B].float()
    fine_emb   = fine_emb[:B].float()

    # Coarse levels → PatchState list (PCA inverse or raw)
    states = [build_patch_states(coarse_emb[b], coarse_level_dims, device,
                                 pca_models=pca_models, n_patches_list=coarse_n_patches)
              for b in range(B)]
    all_levels = batch_patch_states(states)

    # Fine levels
    offset = 0
    for j, li in enumerate(fine_levels_idx):
        d = fine_level_dims[j]
        if pca_models is not None and li in pca_models:
            n_patches = d // pca_models[li].n_components_
            n_comp    = pca_models[li].n_components_
            flat      = fine_emb[:, offset:offset+d].reshape(B * n_patches, n_comp).cpu().numpy()
            flat_full = pca_models[li].inverse_transform(flat)
        else:
            assert fine_n_patches is not None, f"fine_n_patches required for raw fine level {li}"
            n_patches = fine_n_patches[j]
            n_comp    = d // n_patches
            flat_full = fine_emb[:, offset:offset+d].reshape(B * n_patches, n_comp).cpu().numpy()
        emb = torch.tensor(flat_full, dtype=torch.float32).reshape(B, n_patches, -1).to(device)
        pos = make_grid_pos(n_patches, device)
        all_levels.append(PatchState(emb=emb, pos=pos,
                                     non_empty=torch.ones(B, n_patches, device=device),
                                     mae_mask=torch.ones(n_patches, device=device)))
        offset += d

    enc_out = build_enc_out(all_levels)
    recons  = decoder(enc_out)
    return binarize(recons)


In [ ]:
#| export
def make_warmup_cosine_scheduler(optimizer, n_epochs, warmup_epochs=200, eta_min=1e-6):
    """Linear warmup then cosine decay to eta_min (no restarts).

    Inspired by torchcfm image experiments. warmup_epochs: epochs of linear ramp-up.
    After warmup, LR decays as a cosine from base_lr → eta_min over the remaining epochs.
    """
    base_lr = optimizer.param_groups[0]['lr']
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return max(epoch, 1) / max(warmup_epochs, 1)
        progress = (epoch - warmup_epochs) / max(1, n_epochs - warmup_epochs)
        cos_val = 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
        return eta_min / base_lr + (1 - eta_min / base_lr) * cos_val
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def make_warmup_cosine_restart_scheduler(optimizer, T_0, T_mult=2, warmup_frac=0.15, eta_min=1e-6):
    """LambdaLR implementing true warm restarts: linear ramp-up → cosine decay per cycle.
    Periods double each restart (T_mult=2): T_0, T_0*2, T_0*4, ...
    warmup_frac: fraction of each cycle spent warming up to peak lr.
    eta_min: minimum lr (as absolute value, not a multiplier)."""
    base_lr = optimizer.param_groups[0]['lr']

    def get_cycle(epoch):
        """Return (cycle index, position within cycle, cycle length)."""
        T_i, T_prev = T_0, 0
        while T_prev + T_i <= epoch:
            T_prev += T_i
            T_i = int(T_i * T_mult)
        return T_prev, T_i   # cycle_start, cycle_length

    def lr_lambda(epoch):
        cycle_start, T_i = get_cycle(epoch)
        T_cur = epoch - cycle_start
        warmup_end = max(1, int(T_i * warmup_frac))
        if T_cur < warmup_end:
            return T_cur / warmup_end                              # linear warmup → 1.0 (base_lr)
        progress = (T_cur - warmup_end) / max(1, T_i - warmup_end)
        cos_val = 0.5 * (1 + math.cos(math.pi * progress))        # 1 → 0
        return eta_min / base_lr + (1 - eta_min / base_lr) * cos_val

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
#| export
def train_flow_conditional(coarse_model, fine_model, dataset, cfg, device='cpu',
                           decoder=None, pca_models=None):
    """Train flow model(s).  Mode is read from cfg.flow.mode:
      'coarse' – train coarse_model only (unconditional CrossLevelFlowModel)
      'fine'   – train fine_model with frozen coarse as conditioning signal (default)
                 if cfg.flow.fine_teacher_forcing=True, uses real coarse data as x_cond
      'both'   – train both jointly: separate losses, teacher-forcing for fine conditioning
    All hyperparameters read from cfg.flow.  Manages W&B init/finish internally.
    decoder, pca_models: optional piano-roll viz (fine/both only).
    Eval/logging is scoped to the model(s) being trained:
      mode='coarse' → coarse metrics only; mode='fine' → fine metrics only; mode='both' → all.
    """
    from midi_rae.utils import save_checkpoint, load_checkpoint, EMAModel
    from midi_rae.data import ConditionalFlowChunkSampler, ConditionalFlowDataset
    from hydra.core.hydra_config import HydraConfig
    from torchvision.utils import make_grid

    cjprint(f"config file: {HydraConfig.get().job.config_name}\nconfig: {cfg}\ndevice = {device}", color="green")

    fc   = cfg.flow
    mode = fc.get('mode', 'fine')
    assert mode in ('coarse', 'fine', 'both'), f"cfg.flow.mode must be coarse/fine/both; got {mode!r}"
    do_fine = mode in ('fine', 'both')
    fine_teacher_forcing = fc.get('fine_teacher_forcing', False)
    use_amp_fine         = fc.get('use_amp_fine', True) and device != 'cpu'
    if do_fine:
        # Fine model inter-level attention has seq_len=5 (1 parent + 4 children).
        # With bfloat16 AMP, SDPA selects Flash attention which fails at seq_len<32.
        # Disable Flash globally; MATH backend handles arbitrary seq/head dims.
        torch.backends.cuda.enable_flash_sdp(False)

    # --- Hyperparameters ---
    n_epochs             = fc.n_epochs
    lr                   = fc.lr
    if mode == 'coarse': batch_size = fc.get('coarse_batch_size', fc.batch_size)
    elif mode == 'fine': batch_size = fc.get('fine_batch_size',   fc.batch_size)
    else:                batch_size = fc.get('fine_batch_size',   fc.batch_size)  # both: fine is limiting
    warp_s               = fc.warp_s
    time_schedule        = fc.get('time_schedule', 'warp')
    sine_kappa           = fc.get('sine_kappa', 0.0)
    save_every           = fc.get('save_every', 10)
    viz_every            = fc.get('viz_every', 10)
    eval_every           = min(viz_every, fc.get('eval_every', 10))
    eval_n_samples       = fc.get('eval_n_samples', 4000)
    steps_per_epoch      = fc.get('steps_per_epoch', None)
    lr_warmup_epochs     = fc.get('lr_warmup_epochs', 200)
    grad_clip            = fc.get('grad_clip', 1.0)
    ema_eta              = fc.get('ema_eta', 0.9999)
    ema_warmup           = fc.get('ema_warmup', True)
    ema_start_epoch      = fc.get('ema_start_epoch', 0)
    repair_every         = fc.get('repair_every', 1)
    n_repair_projections = fc.get('n_repair_projections', 1)
    repair_chunk_size    = fc.get('repair_chunk_size', None)
    pairing_method          = fc.get('pairing_method', 'ann')  # 'ann' | 'exact' (POT emd) | 'none'
    fine_structured_source  = fc.get('fine_structured_source', False)
    fine_noise_sigma        = fc.get('fine_noise_sigma', 0.1)
    all_source_scales    = list(fc.get('source_scales', [])) or None
    fine_levels          = list(fc.get('fine_levels', [4, 5]))
    fine_level_names     = [f'L{l}' for l in fine_levels]
    fine_levels_idx      = fine_levels

    raw_fn = fc.get('fine_n_components', None)
    fine_n_components = (list(raw_fn) if hasattr(raw_fn, '__iter__') else int(raw_fn)) if raw_fn is not None else None
    fn_list = (fine_n_components if isinstance(fine_n_components, list)
               else [fine_n_components] * len(fine_levels)) if fine_n_components else None

    checkpoint = os.path.expandvars(os.path.expanduser(fc.get('checkpoint', '') or '')) or None

    use_wandb = not cfg.get('no_wandb', False) and hasattr(cfg.wandb, 'flow_project')
    if use_wandb:
        wandb.init(project=cfg.wandb.flow_project, config=dict(fc))
        wandb.define_metric("epoch")
        wandb.define_metric("*", step_metric="epoch")
        if hasattr(cfg, 'tag'): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    # --- Model setup ---
    # In fine mode, keep coarse on CPU: frozen and only needed for piano_rolls_gen viz
    coarse_device = 'cpu' if mode == 'fine' else device
    coarse_model = coarse_model.to(coarse_device)
    ema_kw = dict(eta=ema_eta, update_every=1, dtype=torch.float32,
                  eta_warmup_steps=(1 if ema_warmup else 0))
    coarse_ema   = EMAModel(coarse_model, **ema_kw)
    if mode == 'coarse':
        coarse_model.train()
        fine_ema  = None
        optimizer = optim.Adam(coarse_model.parameters(), lr=lr)
    elif mode == 'fine':
        coarse_model.eval()
        for p in coarse_model.parameters(): p.requires_grad_(False)
        fine_model = fine_model.to(device)
        fine_ema   = EMAModel(fine_model, **ema_kw)
        optimizer  = optim.Adam(fine_model.parameters(), lr=lr)
    else:  # both: both models train; fine conditioned on real_coarse (teacher forcing)
        coarse_model.train()
        fine_model = fine_model.to(device)
        fine_model.train()
        fine_ema  = EMAModel(fine_model, **ema_kw)
        optimizer = optim.Adam(
            list(coarse_model.parameters()) + list(fine_model.parameters()), lr=lr)

    # --- DataLoader ---
    use_fine_pca = getattr(dataset, '_use_fine_pca', False)
    if isinstance(dataset, ConditionalFlowDataset) and not use_fine_pca:
        sampler = ConditionalFlowChunkSampler(dataset)
        dl = DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                        num_workers=0, pin_memory=(device != 'cpu'), drop_last=True)
    else:
        dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
    dl_iter = None
    _steps  = steps_per_epoch or len(dl)

    loss_fn   = nn.MSELoss()
    scheduler = make_warmup_cosine_scheduler(
        optimizer, n_epochs=n_epochs, warmup_epochs=lr_warmup_epochs, eta_min=1e-6)

    epoch_start = 1
    global_step = 0
    real_scatter_logged = False
    scatter_pca_cache   = {}   # keyed by level name; populated on first viz, reused thereafter

    # --- Checkpoint resume ---
    if checkpoint:
        resume_model = coarse_model if mode == 'coarse' else fine_model
        resume_ema   = coarse_ema   if mode == 'coarse' else fine_ema
        resume_model, ckpt = load_checkpoint(resume_model, checkpoint, return_all=True)
        if 'optimizer_state_dict' in ckpt: optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_start = ckpt['epoch'] + 1  # next epoch after the completed one
        global_step = (epoch_start - 1) * _steps
        for _ in range(epoch_start - 1): scheduler.step()
        # Try to load the matching EMA checkpoint (EMAModel_<tag>ckpt_epochN.pt)
        import re as _re
        ema_ckpt_path = _re.sub(r'^(.*/)?[^_/]+_(.*)', r'\1EMAModel_\2', checkpoint)
        if os.path.exists(ema_ckpt_path):
            resume_ema.ema, _ = load_checkpoint(resume_ema.ema, ema_ckpt_path, return_all=True)
            print(f"Resumed EMA from {ema_ckpt_path}")
        else:
            resume_ema.ema.load_state_dict(resume_model.state_dict())
            print(f"EMA checkpoint not found at {ema_ckpt_path}; initialized from model weights")
        print(f"Resumed from {checkpoint} (completed epoch {epoch_start - 1}, starting epoch {epoch_start})")
    print(f"epoch_start = {epoch_start}")

    # --- Dataset dims ---
    coarse_level_dims    = dataset.coarse_level_dims
    coarse_level_names   = [f'L{i}' for i in range(len(coarse_level_dims))]
    coarse_dim           = dataset.coarse.shape[1]
    fine_level_dims      = dataset.fine_level_dims if do_fine else None
    n_coarse             = len(coarse_level_dims)
    coarse_source_scales = all_source_scales[:n_coarse] if all_source_scales else None
    fine_source_scales   = all_source_scales[n_coarse:]  if all_source_scales else None

    def _get_eval_fine(n=2000):
        if getattr(dataset, '_use_fine_pca', False): return dataset.fine[:n].float()
        dataset._load_fine_chunk(0)
        return dataset._fine_chunk_data[:n].float()

    def _gen_chunked_conditional(eval_c, eval_f, n_total):
        """Generate n_total samples in batch_size chunks, accumulate on CPU."""
        c_chunks, f_chunks = [], []
        remaining = n_total
        while remaining > 0:
            chunk_n = min(batch_size, remaining)
            _gc, _gf = generate_samples_conditional(
                eval_c, eval_f,
                n_samples=chunk_n, coarse_dim=coarse_dim,
                target_dims=fine_level_dims, device=device,
                n_steps=20, warp_s=warp_s,
                coarse_level_dims=coarse_level_dims,
                coarse_source_scales=coarse_source_scales,
                fine_source_scales=fine_source_scales,
                use_diffeq_fine=True)
            c_chunks.append(_gc.cpu())
            f_chunks.append(_gf.cpu())
            remaining -= chunk_n
        return torch.cat(c_chunks, dim=0), torch.cat(f_chunks, dim=0)

    def _gen_chunked_coarse(eval_c, n_total):
        """Generate n_total coarse samples in batch_size chunks, accumulate on CPU."""
        chunks = []
        remaining = n_total
        while remaining > 0:
            chunk_n = min(batch_size, remaining)
            chunks.append(generate_samples(eval_c, n_samples=chunk_n,
                                           dim=coarse_dim, device=device,
                                           n_steps=20, warp_s=warp_s,
                                           source_scales=coarse_source_scales,
                                           level_dims=coarse_level_dims).cpu())
            remaining -= chunk_n
        return torch.cat(chunks, dim=0)

    def _gen_chunked_tf(eval_f, n_total):
        """Generate fine samples conditioned on real coarse data (teacher-forcing eval).
        Uses real coarse embeddings from the dataset — no coarse model inference needed.
        Gives the best-case quality estimate of the fine model in isolation."""
        eval_f.eval()
        f_chunks = []
        remaining, offset = n_total, 0
        while remaining > 0:
            chunk_n = min(batch_size, remaining)
            cond = dataset.coarse[offset:offset+chunk_n].float().to(device)
            y    = sample_source((chunk_n, sum(fine_level_dims)), device=device,
                                  source_scales=fine_source_scales, level_dims=fine_level_dims)
            ts   = warp_time(torch.linspace(0, 1, 21), s=warp_s).to(device)
            with torch.no_grad():
                for i in range(20):
                    dt = (ts[i+1] - ts[i]).item()
                    t_ = torch.full((chunk_n, 1), ts[i].item(), device=device)
                    y  = y + eval_f(y, t_, cond) * dt
            f_chunks.append(y.cpu())
            offset    += chunk_n
            remaining -= chunk_n
        return torch.cat(f_chunks, dim=0)

    # --- Training loop ---
    for epoch in range(epoch_start, n_epochs + 1):
        if mode == 'fine': coarse_model.eval()
        else:              coarse_model.train()
        if do_fine: fine_model.train()

        epoch_loss = 0.
        if steps_per_epoch:
            if dl_iter is None:
                import itertools; dl_iter = itertools.cycle(dl)
            batches = (next(dl_iter) for _ in range(_steps))
        else:
            batches = dl
        pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch}/{n_epochs} [{mode}]', leave=False)

        for real_coarse, real_fine in pbar:
            real_coarse = real_coarse.to(device)
            B = real_coarse.size(0)
            t = sample_time((B, 1), schedule=time_schedule, warp_s=warp_s, sine_kappa=sine_kappa, device=device)

            noise_coarse = sample_source((B, real_coarse.size(1)), device=device,
                                         source_scales=coarse_source_scales,
                                         level_dims=coarse_level_dims)
            if do_fine:
                real_fine  = real_fine.to(device)
                if fine_structured_source:
                    # Build structured source: for each fine level, tile the next-coarser
                    # level's PCA embeddings spatially (×4 children per patch) and pad
                    # remaining dims with small Gaussian noise.
                    noise_parts = []
                    offset = 0
                    prev_pca = None   # running "coarser" patch embeddings [B, n_prev, d_prev]
                    for j, (np_j, nc_j) in enumerate(
                            zip(dataset.fine_n_patches, dataset.fine_n_comp)):
                        d_j = np_j * nc_j
                        if prev_pca is None:
                            # L3 (coarsest fine): pure noise
                            noise_parts.append(torch.randn(B, d_j, device=device))
                        else:
                            # tile prev_pca from n_prev → n_prev*4 patches
                            n_prev, d_prev = prev_pca.shape[1], prev_pca.shape[2]
                            tiled = prev_pca.unsqueeze(2).expand(
                                B, n_prev, 4, d_prev).reshape(B, np_j, d_prev)
                            if d_prev < nc_j:
                                pad = torch.randn(B, np_j, nc_j - d_prev,
                                                  device=device) * fine_noise_sigma
                                src = torch.cat([tiled, pad], dim=2)
                            else:
                                src = tiled[:, :, :nc_j]
                            noise_parts.append(src.reshape(B, d_j))
                        # save current level's real PCA (from the fine batch) for next level
                        real_slice = real_fine[:, offset:offset+d_j].reshape(B, np_j, nc_j)
                        prev_pca = real_slice.detach()  # use real data during training
                        offset += d_j
                    noise_fine = torch.cat(noise_parts, dim=1)
                else:
                    noise_fine = sample_source((B, real_fine.size(1)), device=device,
                                               source_scales=fine_source_scales,
                                               level_dims=fine_level_dims)

            if repair_every and global_step % repair_every == 0:
                noise_coarse, real_coarse = do_pairing(noise_coarse, real_coarse,
                                                        method=pairing_method,
                                                        n_projections=n_repair_projections,
                                                        chunk_size=repair_chunk_size)
                if do_fine:
                    noise_fine, real_fine = do_pairing(noise_fine, real_fine,
                                                        method=pairing_method,
                                                        n_projections=n_repair_projections,
                                                        chunk_size=repair_chunk_size)

            x_t_coarse      = (1 - t) * noise_coarse + t * real_coarse
            v_coarse_target = real_coarse - noise_coarse

            optimizer.zero_grad()
            if mode == 'coarse':
                loss = loss_fn(coarse_model(x_t_coarse, t), v_coarse_target)

            elif mode == 'fine':
                x_t_fine      = (1 - t) * noise_fine + t * real_fine
                v_fine_target = real_fine - noise_fine
                if fine_teacher_forcing:
                    x_cond = real_coarse
                else:
                    with torch.no_grad():
                        v_coarse = coarse_model(x_t_coarse, t)
                        x_cond   = x_t_coarse + (1 - t) * v_coarse
                #with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_amp_fine):
                loss = loss_fn(fine_model(x_t_fine, t, x_cond), v_fine_target)

            else:  # both: separate losses, teacher-forcing conditioning for fine
                x_t_fine      = (1 - t) * noise_fine + t * real_fine
                v_fine_target = real_fine - noise_fine
                coarse_loss   = loss_fn(coarse_model(x_t_coarse, t), v_coarse_target)
                #with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_amp_fine):
                fine_loss = loss_fn(fine_model(x_t_fine, t, real_coarse), v_fine_target)
                loss          = coarse_loss + fine_loss

            loss.backward()
            if grad_clip > 0:
                clip_params = (list(coarse_model.parameters()) if mode != 'fine' else []) + \
                              (list(fine_model.parameters()) if do_fine else [])
                torch.nn.utils.clip_grad_norm_(clip_params, max_norm=grad_clip)
            optimizer.step()
            if mode != 'fine': coarse_ema.update(coarse_model)
            if do_fine:        fine_ema.update(fine_model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            global_step += 1

        avg_loss = epoch_loss / _steps
        cur_lr   = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch}/{n_epochs}  loss={avg_loss:.4f}  lr={cur_lr:.2e}  [{mode}]')
        log_dict = {'train/loss': avg_loss, 'train/lr': cur_lr, 'epoch': epoch}

        if eval_every and epoch % eval_every == 0:
            use_ema     = epoch >= ema_start_epoch
            eval_coarse = coarse_ema.ema if (mode != 'fine' and use_ema) else coarse_model
            eval_fine   = fine_ema.ema   if (do_fine and use_ema)        else (fine_model if do_fine else None)
            real_coarse_eval = dataset.coarse[:eval_n_samples].float()

            if do_fine:
                real_fine_eval = _get_eval_fine(eval_n_samples)
                # Metrics use teacher-forced fine samples: best-case eval (real coarse conditioning)
                tf_fine_cpu = _gen_chunked_tf(eval_fine, eval_n_samples)
                fine_metrics = eval_flow(eval_fine, real_fine_eval, n_samples=eval_n_samples,
                                          level_dims=fine_level_dims, gen=tf_fine_cpu,
                                          level_names=fine_level_names,
                                          level_n_patches=dataset.fine_n_patches)
                log_dict.update({f'eval/{k}': v for k, v in fine_metrics.items()})
            else:
                gen_coarse_cpu = _gen_chunked_coarse(eval_coarse, eval_n_samples)

            # Coarse metrics: only when coarse model is actually being trained
            if mode != 'fine':
                coarse_metrics = eval_flow(eval_coarse, real_coarse_eval, n_samples=eval_n_samples,
                                            level_dims=coarse_level_dims, gen=gen_coarse_cpu,
                                            level_names=coarse_level_names,
                                            level_n_patches=dataset.coarse_n_patches)
                log_dict.update({f'eval/{k}': v for k, v in coarse_metrics.items()})

            if (wandb.run is not None) and viz_every and epoch % viz_every == 0:
                if do_fine:
                    real_scatter_logged = _wandb_log_viz(
                        log_dict, eval_fine, real_fine_eval, fine_level_dims, epoch,
                        real_scatter_logged, gen=tf_fine_cpu, level_names=fine_level_names,
                        level_n_components=dataset.fine_n_comp, pca_cache=scatter_pca_cache)
                if mode != 'fine':
                    _wandb_log_viz(log_dict, eval_coarse, real_coarse_eval, coarse_level_dims,
                                   epoch, False, gen=gen_coarse_cpu, level_names=coarse_level_names,
                                   level_n_components=dataset.coarse_n_comp, pca_cache=scatter_pca_cache)
                if do_fine and decoder is not None:
                    n_rolls = 64
                    _dec_kw = dict(pca_models=pca_models,
                                   coarse_n_patches=dataset.coarse_n_patches,
                                   fine_n_patches=dataset.fine_n_patches)
                    # piano_rolls_real: decode real embeddings directly (upper-bound quality check)
                    real_rolls = decode_flow_to_piano_rolls(
                        real_coarse_eval[:n_rolls].cpu(), real_fine_eval[:n_rolls].cpu(),
                        coarse_level_dims=coarse_level_dims, fine_level_dims=fine_level_dims,
                        fine_levels_idx=fine_levels_idx, cfg=cfg, decoder=decoder,
                        device=device, n_samples=n_rolls, **_dec_kw)
                    real_grid = make_grid(real_rolls[:n_rolls], nrow=8, normalize=True)
                    log_dict['media/piano_rolls_real'] = wandb.Image(real_grid, caption=f'Real Epoch {epoch}')
                    # piano_rolls_gen: loop over EMA and normal models for comparison
                    for net_tag, c_model, f_model in [('ema', eval_coarse, eval_fine),
                                                       ('normal', coarse_model, fine_model if do_fine else None)]:
                        c_model.to(device)
                        gen_coarse_64, gen_fine_64 = generate_samples_conditional(
                            c_model, f_model,
                            n_samples=n_rolls, coarse_dim=coarse_dim,
                            target_dims=fine_level_dims, device=device,
                            n_steps=20, warp_s=warp_s,
                            coarse_level_dims=coarse_level_dims,
                            coarse_source_scales=coarse_source_scales,
                            fine_source_scales=fine_source_scales,
                            use_diffeq_fine=True)
                        c_model.to('cpu')
                        gen_coarse_64, gen_fine_64 = gen_coarse_64.cpu(), gen_fine_64.cpu()
                        rolls = decode_flow_to_piano_rolls(
                            gen_coarse_64, gen_fine_64,
                            coarse_level_dims=coarse_level_dims, fine_level_dims=fine_level_dims,
                            fine_levels_idx=fine_levels_idx, cfg=cfg, decoder=decoder,
                            device=device, n_samples=n_rolls, **_dec_kw)
                        grid = make_grid(rolls[:n_rolls], nrow=8, normalize=True)
                        log_dict[f'media/piano_rolls_gen_{net_tag}'] = wandb.Image(grid, caption=f'{net_tag} Epoch {epoch}')
                        if fine_teacher_forcing:
                            # Teacher-forcing viz: integrate fine conditioned on real coarse
                            idx_tf = torch.randperm(len(dataset.coarse))[:n_rolls]
                            tf_coarse = dataset.coarse[idx_tf].to(device)
                            tf_fine_noise = sample_source((n_rolls, sum(fine_level_dims)), device=device,
                                                           source_scales=fine_source_scales,
                                                           level_dims=fine_level_dims)
                            ts_tf = warp_time(torch.linspace(0, 1, 21), s=warp_s)
                            y_fine_tf = tf_fine_noise
                            with torch.no_grad():
                                for i in range(20):
                                    dt_tf = (ts_tf[i+1] - ts_tf[i]).item()
                                    t_tf  = torch.full((n_rolls, 1), ts_tf[i].item(), device=device)
                                    y_fine_tf = y_fine_tf + f_model(y_fine_tf, t_tf, tf_coarse) * dt_tf
                            tf_rolls = decode_flow_to_piano_rolls(
                                tf_coarse.cpu(), y_fine_tf.cpu(),
                                coarse_level_dims=coarse_level_dims, fine_level_dims=fine_level_dims,
                                fine_levels_idx=fine_levels_idx, cfg=cfg, decoder=decoder,
                                device=device, n_samples=n_rolls, **_dec_kw)
                            tf_grid = make_grid(tf_rolls[:n_rolls], nrow=8, normalize=True)
                            log_dict[f'media/piano_rolls_tf_{net_tag}'] = wandb.Image(tf_grid, caption=f'TF {net_tag} Epoch {epoch}')
                gc.collect()

            if mode == 'fine': coarse_model.eval()
            else:              coarse_model.train()
            if do_fine: fine_model.train()

        if (wandb.run is not None): wandb.log(log_dict, step=global_step)

        if mode == 'coarse':
            save_checkpoint((coarse_ema, coarse_model), epoch, avg_loss, cfg or {},
                             optimizer=optimizer, save_every=save_every, tag=cfg.tag)
        elif mode == 'fine':
            save_checkpoint((fine_ema, fine_model), epoch, avg_loss, cfg or {},
                             optimizer=optimizer, save_every=save_every, tag=cfg.tag)
        else:
            save_checkpoint((coarse_ema, coarse_model), epoch, avg_loss, cfg or {},
                             optimizer=optimizer, save_every=save_every, tag=cfg.tag + '_coarse')
            save_checkpoint((fine_ema, fine_model), epoch, avg_loss, cfg or {},
                             optimizer=optimizer, save_every=save_every, tag=cfg.tag + '_fine')
        scheduler.step()

    if (wandb.run is not None): wandb.finish()
    print(f"FINISHED. Best loss: {save_checkpoint.best_val_loss:.6f}")


In [ ]:
#| eval: false
import hydra
from omegaconf import DictConfig

def _run_flow(cfg: DictConfig):
    """Build coarse and (if needed) fine model, then call train_flow_conditional."""
    from midi_rae.data import ConditionalFlowDataset
    from midi_rae.utils import load_checkpoint
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"device = {device}")

    fc      = cfg.flow
    mode    = fc.get('mode', 'fine')
    do_fine = mode in ('fine', 'both')
    fine_levels = list(fc.get('fine_levels', [4, 5]))

    fine_pca_dir      = fc.get('fine_pca_dir', None)
    fine_n_components = fc.get('fine_n_components', None)
    if fine_pca_dir:
        fine_pca_dir = os.path.expandvars(os.path.expanduser(str(fine_pca_dir)))
        fine_n_components = (list(fine_n_components) if hasattr(fine_n_components, '__iter__')
                             else int(fine_n_components)) if fine_n_components is not None else None

    dataset = ConditionalFlowDataset(
        pca_dir           = fc.pca_dir,
        encoded_dir       = fc.get('encoded_dir', None),
        pca_levels        = list(fc.get('pca_levels', ['L0', 'L1', 'L2'])),
        fine_levels       = fine_levels,
        emb_key           = fc.get('emb_key', 'emb1'),
        fine_pca_dir      = fine_pca_dir,
        fine_n_components = fine_n_components,
        fine_raw          = fc.get('fine_raw', False),
    )
    print(f"  {len(dataset)} samples  coarse_dims={dataset.coarse_level_dims}  fine_dims={dataset.fine_level_dims}")

    coarse_level_dims = dataset.coarse_level_dims
    coarse_n_comp     = dataset.coarse_n_comp   # inferred from data shape — no config needed

    coarse_model = CrossLevelFlowModel(
        level_dims    = coarse_level_dims,
        level_n_comp  = coarse_n_comp,
        h_dim         = fc.get('coarse_h_dim', fc.h_dim),
        n_layers      = fc.get('coarse_n_layers', fc.n_layers),
        n_attn_layers = fc.get('n_attn_layers', 2),
        n_heads       = fc.get('n_heads', 8),
        t_dim         = fc.get('t_dim', 64),
    )
    n_params = sum(p.numel() for p in coarse_model.parameters())
    print(f"  CrossLevelFlowModel: {n_params:,} parameters  patch_counts={coarse_model.level_n_patches}")
    coarse_ckpt = fc.get('coarse_ckpt', None)
    if coarse_ckpt and str(coarse_ckpt).lower() not in ('none', 'null', ''):
        coarse_ckpt  = os.path.expandvars(os.path.expanduser(str(coarse_ckpt)))
        coarse_model = load_checkpoint(coarse_model, coarse_ckpt)
        print(f"  Loaded coarse model from {coarse_ckpt}")
    elif mode != 'coarse':
        print("  WARNING: coarse_ckpt not set — coarse model starts from random weights")

    fine_model = None
    if do_fine:
        fine_n_comp = dataset.fine_n_comp  # inferred from data shape
        fine_model_type = fc.get('fine_model_type', 'mlp')
        if fine_model_type == 'unet':
            fine_model = UNetFineFlowModel(
                cond_dims      = coarse_level_dims,
                target_dims    = dataset.fine_level_dims,
                target_n_comp  = fine_n_comp,
                h_dim          = fc.h_dim,
                t_dim          = fc.get('t_dim', 64),
            )
            n_params = sum(p.numel() for p in fine_model.parameters())
            print(f"  UNetFineFlowModel: {n_params:,} parameters")
        else:
            fine_model = ConditionalFineFlowModel(
                cond_dims             = coarse_level_dims,
                target_dims           = dataset.fine_level_dims,
                target_n_comp         = fine_n_comp,
                cond_n_comp           = coarse_n_comp,
                h_dim                 = fc.h_dim,
                n_layers              = fc.n_layers,
                t_dim                 = fc.get('t_dim', 64),
                n_fine_attn_layers    = fc.get('fine_attn_layers', 2),
                n_lateral_attn_layers = fc.get('fine_lateral_attn_layers', 0),
                grad_checkpoint       = fc.get('grad_checkpoint', False),
            )
            n_params = sum(p.numel() for p in fine_model.parameters())
            print(f"  ConditionalFineFlowModel: {n_params:,} parameters")

    decoder, pca_models = None, None
    if do_fine:
        decoder_ckpt = os.path.expandvars(os.path.expanduser(str(cfg.generate.get('decoder_ckpt', '') or '')))
        if decoder_ckpt:
            from pathlib import Path
            from midi_rae.swin import SwinDecoder
            from midi_rae.train_dec import load_pca_models
            m = cfg.model
            decoder = SwinDecoder(
                img_height=cfg.data.image_size, img_width=cfg.data.image_size,
                patch_h=m.patch_h, patch_w=m.patch_w, out_channels=cfg.data.in_channels,
                embed_dim=m.embed_dim, depths=list(m.dec_depths),
                num_heads=list(m.dec_num_heads), window_size=m.window_size,
                mlp_ratio=m.mlp_ratio, drop_path_rate=0.0)
            decoder = load_checkpoint(decoder, decoder_ckpt).to(device).eval()
            for p in decoder.parameters(): p.requires_grad_(False)
            if getattr(dataset, '_use_fine_pca', False):
                pca_dir    = Path(os.path.expandvars(os.path.expanduser(str(fc.pca_dir))))
                pca_models = load_pca_models(str(pca_dir), len(coarse_level_dims) + len(fine_levels))
                print(f"  Loaded decoder + {len(pca_models)} PCA models for piano roll viz")
            else:
                print("  Decoder loaded; raw mode — pca_models=None for viz")

    if fc.get('compile', False):
        import midi_rae.train_flow as _tf
        print("  torch.compile: compiling flow model(s) and ann_repair...")
        coarse_model  = torch.compile(coarse_model)
        if fine_model is not None:
            fine_model = torch.compile(fine_model, backend='aot_eager')
        _tf.ann_repair = torch.compile(_tf.ann_repair)
        print("  torch.compile: done")

    train_flow_conditional(coarse_model, fine_model, dataset, cfg,
                           device=device, decoder=decoder, pca_models=pca_models)


@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def train_flow_main(cfg: DictConfig):
    _run_flow(cfg)

if __name__ == '__main__':
    train_flow_main()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()